# Piezo Triangle Scan - Beam Position Analysis

Loads a CSV produced by `piezo_triangle_scan.py` (columns: sample, timestamp,
voltage_setpoint_V, voltage_readback_V, centroid_x_um, centroid_y_um,
gaussfit_center_x_um, gaussfit_center_y_um, gaussfit_rating_x, gaussfit_rating_y,
peak_intensity_counts, total_power_mW) and plots:

- voltage and beam **Gaussian-fit center** vs. sample (time series)
- **driven-axis displacement vs. piezo voltage** (the mirror's transfer
  function), colored by sample order so any hysteresis between the rising and
  falling ramps of the triangle wave is visible
- a linear fit to estimate the driven-axis sensitivity in um/V

The driven axis (X or Y) is read from the filename's `_axisX`/`_axisY` token,
and all displacement plots use the Gaussian-fit center rather than the plain
centroid.


In [ ]:
import glob
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

DATA_DIR = "."

# pick the most recent scan CSV by default; override CSV_PATH to load a specific file
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "piezo_scan_*.csv")))
CSV_PATH = csv_files[-1] if csv_files else None
# CSV_PATH = "piezo_scan_20260625_164729.csv"
print("Available scans:", [os.path.basename(f) for f in csv_files])
print("Loading:", CSV_PATH)

# which piezo axis was driven is encoded in the filename as "_axisX" / "_axisY"
# (added by piezo_triangle_scan.py). Fall back to Y for older files without it.
_m = re.search(r"_axis([XY])", os.path.basename(CSV_PATH or ""), re.IGNORECASE)
AXIS = _m.group(1).upper() if _m else "Y"
if _m is None:
    print("No axis token in filename; defaulting to AXIS =", AXIS)
else:
    print("Scanned axis (from filename):", AXIS)

# always plot the Gaussian-fit center (not the plain centroid) for the driven axis
POS_COL = f"gaussfit_center_{AXIS.lower()}_um"

# The camera is mounted rotated 90 deg, so the camera's reported X axis is the
# physical Y axis and vice versa. Data columns are left exactly as the camera
# reports them; only the axis *labels* are swapped for display.
DISP = {"X": "Y", "Y": "X"}
AXIS_LABEL = DISP[AXIS]

In [15]:
df = pd.read_csv(CSV_PATH, parse_dates=["timestamp"])
df.head()

,sample,timestamp,voltage_setpoint_V,voltage_readback_V,centroid_x_um,centroid_y_um,gaussfit_center_x_um,gaussfit_center_y_um,gaussfit_rating_x,gaussfit_rating_y,peak_intensity_counts,total_power_mW
0,0,2026-07-02 10:26:04.079397,0.0,0.00,-158.049797,-149.932182,1680.791451,-1209.615501,0.973445,0.971521,24.0,0.000007
1,1,2026-07-02 10:26:04.894956,1.0,0.00,-181.133427,-134.302231,1661.955594,-1204.363656,0.975853,0.968859,21.0,0.000007
2,2,2026-07-02 10:26:05.311453,2.0,1.07,-189.120340,-125.457496,1668.949261,-1201.050473,0.975110,0.969250,20.0,0.000007
3,3,2026-07-02 10:26:05.742092,3.0,3.04,-182.429410,-134.470922,1669.160272,-1198.574836,0.975093,0.971214,21.0,0.000007
4,4,2026-07-02 10:26:06.139126,4.0,4.04,-175.031043,-128.854533,1683.778670,-1192.990532,0.974916,0.969596,21.0,0.000007


## Voltage and beam centroid vs. sample

In [ ]:
fig, ax_v = plt.subplots(figsize=(9, 4))
ax_v.plot(df["sample"], df["voltage_readback_V"], color="tab:blue", label="Piezo voltage (readback)")
ax_v.set_xlabel("Sample")
ax_v.set_ylabel("Voltage (V)", color="tab:blue")
ax_v.tick_params(axis="y", labelcolor="tab:blue")

ax_p = ax_v.twinx()
# camera rotated 90 deg: camera X column is physical Y and vice versa (labels swapped)
ax_p.plot(df["sample"], df["gaussfit_center_x_um"], color="tab:red", label=f"Fit center {DISP['X']}")
ax_p.plot(df["sample"], df["gaussfit_center_y_um"], color="tab:orange", label=f"Fit center {DISP['Y']}")
ax_p.set_ylabel("Gaussian fit center displacement (um)", color="tab:red")
ax_p.tick_params(axis="y", labelcolor="tab:red")
ax_p.legend(loc="upper right")

fig.suptitle("Piezo voltage and beam Gaussian-fit center vs. sample")
fig.tight_layout()
plt.show()

## Driven-axis displacement vs. voltage

Each point is colored by sample index (time order), so the rising and
falling legs of the triangle wave are distinguishable - a gap between
the two colors at the same voltage indicates piezo hysteresis. The plotted
displacement is the Gaussian-fit center of the driven axis (from the filename).

In [ ]:
elapsed_s = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(df["voltage_readback_V"], df[POS_COL],
                 c=elapsed_s, cmap="viridis", s=20)
ax.plot(df["voltage_readback_V"], df[POS_COL], "-", color="gray", alpha=0.3, linewidth=0.8)

ax.set_xlabel("Piezo voltage, readback (V)")
ax.set_ylabel(f"Gaussian fit center {AXIS_LABEL} displacement (um)")
ax.set_title(f"Beam {AXIS_LABEL} displacement vs. piezo voltage")

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("Time (s)")

# max spread in displacement among samples that share the same voltage setpoint --
# this is the hysteresis gap between the rising and falling legs of the triangle wave
spread_by_voltage = df.groupby("voltage_setpoint_V")[POS_COL].agg(lambda s: s.max() - s.min())
max_spread = spread_by_voltage.max()
max_spread_voltage = spread_by_voltage.idxmax()

ax.annotate(
    f"Max displacement spread\nat same voltage: {max_spread:.2f} um\n(at {max_spread_voltage:.1f} V)",
    xy=(0.02, 0.02), xycoords="axes fraction", va="bottom", ha="left",
    fontsize=9, bbox=dict(boxstyle="round", fc="white", ec="gray", alpha=0.8),
)

fig.tight_layout()
plt.show()

print(f"Max displacement spread at the same voltage: {max_spread:.3f} um, at setpoint {max_spread_voltage:.2f} V")

## Linear fit: driven-axis sensitivity (um/V)

In [ ]:
slope, intercept = np.polyfit(df["voltage_readback_V"], df[POS_COL], 1)
print(f"{AXIS_LABEL} sensitivity: {slope:.3f} um/V")
print(f"Intercept: {intercept:.3f} um")

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(df["voltage_readback_V"], df[POS_COL], s=15, alpha=0.6, label="Data")

v_fit = np.linspace(df["voltage_readback_V"].min(), df["voltage_readback_V"].max(), 200)
ax.plot(v_fit, slope * v_fit + intercept, color="tab:red",
        label=f"Linear fit: {slope:.3f} um/V")

ax.set_xlabel("Piezo voltage, readback (V)")
ax.set_ylabel(f"Gaussian fit center {AXIS_LABEL} displacement (um)")
ax.set_title(f"{AXIS_LABEL} displacement vs. voltage with linear fit")
ax.legend()
fig.tight_layout()
plt.show()